# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadBilalFarooq/Assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My ML Task Type: Binary Classification

My lane is Lane 1 — Content Refresh Prioritization.

This is a Binary Classification problem:
- Input: page features (position, CTR, word count, age, etc.)
- Output: 1 = this page needs refresh / 0 = this page is fine

Why classification and not ranking?
Because the content team needs a clear YES/NO decision
per page — "should we refresh this page this week or not?"
A ranked list would also work, but binary classification
gives the clearest action signal.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target / Proxy Column: trend_direction

The ideal target would be: "Did refreshing this page
improve its traffic?" — but we don't have that data.

So we use a proxy:
- trend_direction == 'down' → Label 1 (needs refresh)
- trend_direction != 'down' → Label 0 (okay for now)

This is a proxy because:
- A declining page is our best signal that content
  needs attention
- We cannot directly observe "refresh needed" in the data
- This assumption is directional, not causal proof

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## Success Metric: Precision@50

Primary metric: Precision@50
- Of the top 50 pages the model flags for refresh,
  how many are actually declining?
- Baseline (hand-written rule): ~24% (12/50 correct)
- Our target: beat 50% (25/50 correct)

Why not accuracy?
- Dataset is imbalanced — more stable pages than declining
- Accuracy would be high even if model ignores all
  declining pages
- Precision@50 matches the real workflow: content team
  reviews top 50 recommendations per week

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os, sys, subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("✅ Repo clone ho gaya!")

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

✅ Repo clone ho gaya!
Working dir: /content/flyrank-ml-internship-starter


In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Unit of analysis dikhao
print("=== Unit of Analysis ===")
print("Ek row = Ek webpage (ek specific time period mein)")
print(f"\nTotal rows (pages): {len(df)}")
print(f"Total columns (features): {len(df.columns)}")

# Pehli 3 rows dikhao
print("\n=== Sample Rows ===")
print(df.head(3).to_string())

# Important columns
print("\n=== Key Columns ===")
key_cols = ["avg_position", "ctr", "impressions_90d",
            "word_count", "trend_direction", "content_age_days"]
print(df[key_cols].head(5).to_string())

=== Unit of Analysis ===
Ek row = Ek webpage (ek specific time period mein)

Total rows (pages): 30000
Total columns (features): 44

=== Sample Rows ===
             content_id          client_id  search_volume  competition competition_level   cpc     content_type    main_intent  word_count  char_count provider_used              model_used  impressions_90d  clicks_90d  pageviews_90d  sessions_90d  users_90d  engaged_sessions_90d  ai_sessions_90d  scroll_events_90d  days_with_impressions  days_with_sessions  impressions_last_30d  clicks_last_30d  sessions_last_30d  impressions_prev_30d  clicks_prev_30d  sessions_prev_30d  content_age_days age_tier  age_tier_order  days_since_last_update freshness_tier word_count_tier char_count_tier   ctr  avg_position  engagement_rate  scroll_rate  ai_traffic_pct impression_tier position_tier trend_direction  trend_pct
0  content_304f48230142  client_f369cb89fc           10.0         0.67              HIGH  2.05  keyword article  transactional      322

In [6]:
# Target column banana
df["target"] = (df["trend_direction"] == "down").astype(int)

print("=== Target Column Distribution ===")
print(df["target"].value_counts())
print(f"\nClass 1 (needs refresh): {df['target'].sum()} pages")
print(f"Class 0 (okay):          {(df['target']==0).sum()} pages")
print(f"\nRefresh rate: {df['target'].mean()*100:.1f}% of pages are declining")

# Sample with target
print("\n=== Sample with Target Column ===")
cols = ["avg_position", "ctr", "word_count",
        "trend_direction", "target"]
print(df[cols].head(8).to_string())

=== Target Column Distribution ===
target
1    16262
0    13738
Name: count, dtype: int64

Class 1 (needs refresh): 16262 pages
Class 0 (okay):          13738 pages

Refresh rate: 54.2% of pages are declining

=== Sample with Target Column ===
   avg_position   ctr  word_count trend_direction  target
0          10.6  0.76      3221.0            down       1
1          20.3  0.05      2481.0            down       1
2          36.5  0.09      3515.0            down       1
3           6.2  0.49         NaN          stable       0
4          44.0  0.13      2803.0            down       1
5           8.5  0.03      3080.0            down       1
6           7.0  0.00      3059.0            down       1
7          21.2  0.06         NaN          stable       0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## Why ML Beats a Fixed Rule

A fixed rule might look like:
"IF position > 20 AND ctr < 0.02 THEN flag for refresh"

Problems with this rule:
1. It only uses 2 features — ignores word count,
   content age, impressions, etc.
2. The thresholds (20, 0.02) are guesses — not
   learned from data
3. It cannot adapt — different content types may
   need different thresholds

What ML does better:
- Learns from ALL features simultaneously
- Finds non-obvious patterns
  (e.g. high-volume pages with low CTR at position 5)
- Adapts thresholds based on what actually predicts decline
- Notebook 01 proved this:
  Rule = 24% Precision@50
  ML   = 74% Precision@50 (3x better)

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.